# Aula 2 – Vídeo 3: Principais Componentes do LangGraph

Neste notebook, exploramos:
- **Nós**: funções que leem/escrevem o estado
- **Arestas**: conexões (lineares e condicionais)
- **Estado compartilhado**: contrato que circula entre nós
- **Ciclo de vida**: construir → executar → encerrar


## 1) Imports e setup (LLM opcional)
O notebook funciona **mesmo sem** chave da OpenAI; o LLM é opcional.

In [1]:
import os
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import PromptTemplate
except Exception:
    ChatOpenAI = None
    PromptTemplate = None

load_dotenv()
llm = None
if ChatOpenAI:
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    llm = ChatOpenAI(model=model, temperature=0)

## 2) Estado compartilhado
Usamos `TypedDict` para declarar os campos que podem circular entre os nós.

In [2]:
class State(TypedDict, total=False):
    texto: str
    intencao: str
    resumo: str
    palavras: int

## 3) Nós (funções)
Cada nó recebe `State` e devolve um **delta** de estado (um dicionário parcial).

In [3]:
def classificar(s: State) -> State:
    t = (s.get("texto") or "").lower()
    intent = "fatura" if "fatura" in t else "faq"
    return {"intencao": intent}

def resumir(s: State) -> State:
    texto = s.get("texto", "")
    if llm and PromptTemplate:
        prompt = PromptTemplate.from_template("Resuma em uma frase: {texto}")
        chain = prompt | llm
        resumo = chain.invoke({"texto": texto}).content
        return {"resumo": resumo}
    return {"resumo": (texto[:60] + "...") if len(texto) > 60 else texto}

def contar(s: State) -> State:
    return {"palavras": len(s.get("resumo", "").split())}

## 4) Montagem do grafo (arestas lineares e condicionais)
Definimos o ponto de entrada e as arestas — inclusive a **condicional** a partir da intenção.

In [4]:
g = StateGraph(State)
g.add_node("classificar", classificar)
g.add_node("resumir", resumir)
g.add_node("contar", contar)

g.set_entry_point("classificar")

def rota(s: State):
    return s.get("intencao", "faq")

g.add_conditional_edges("classificar", rota, {"faq": "resumir", "fatura": "contar"})
g.add_edge("resumir", "contar")
g.add_edge("contar", END)

app = g.compile()
print(app.get_graph().draw_ascii())

          +-----------+     
          | __start__ |     
          +-----------+     
                *           
                *           
                *           
        +-------------+     
        | classificar |     
        +-------------+     
          ..         ..     
        ..             .    
       .                ..  
+---------+               . 
| resumir |             ..  
+---------+            .    
          **         ..     
            **     ..       
              *   .         
           +--------+       
           | contar |       
           +--------+       
                *           
                *           
                *           
          +---------+       
          | __end__ |       
          +---------+       


## 5) Execução – exemplos
Executamos dois casos para ver a bifurcação: (a) caminho *faq* e (b) caminho *fatura*.

In [5]:
print("\n=== Execução (FAQ) ===")
estado1 = {"texto": "O que é LangGraph e como ele funciona?"}
print(app.invoke(estado1))

print("\n=== Execução (Fatura) ===")
estado2 = {"texto": "Quero ver a fatura de setembro"}
print(app.invoke(estado2))


=== Execução (FAQ) ===
{'texto': 'O que é LangGraph e como ele funciona?', 'intencao': 'faq', 'resumo': 'LangGraph é uma plataforma que utiliza grafos de conhecimento para melhorar a compreensão e a geração de linguagem natural, conectando informações de forma estruturada para facilitar a interação entre humanos e máquinas.', 'palavras': 32}

=== Execução (Fatura) ===
{'texto': 'Quero ver a fatura de setembro', 'intencao': 'fatura', 'palavras': 0}
